# Does trade promotion pay for itself?

### Price, promotion and basket economics across 77 US grocery stores, 2009–2012

**This notebook — time series and cohort analysis.** The panel established in [`0_prelim_analysis.ipynb`](0_prelim_analysis.ipynb) and
[`1_distribution_analysis.ipynb`](1_distribution_analysis.ipynb) is a weekly
**store x SKU** grid. Those two notebooks settled shape and quality; this one asks
what the *time axis* and *groups* add:

1. **Is the weekly series moving?** — level, trend, and the units/spend divergence.
2. **Is there a seasonal cycle, and does it repeat year to year?** — decomposition,
   then the same ISO week in 2010 against 2011.
3. **Do store strata behave differently over time?** — segment, MSA and size cohorts.
4. **Which metrics can we actually build here?** — elasticity, promotion ROI proxy,
   basket economics, active-household proxy, distribution penetration.

> **The constraint that shapes every question above.** There is no customer
> identifier anywhere in this dataset. `dh Transaction Data` carries
> `WEEK_END_DATE, STORE_NUM, UPC, UNITS, VISITS, HHS, SPEND, PRICE, BASE_PRICE,
> FEATURE, DISPLAY, TPR_ONLY` — twelve columns, none of them a customer key.
> `HHS` and `VISITS` are *counts* per store-week-SKU, not IDs, so no household can
> be followed across weeks. Customer-level LTV, DAU/MAU, retention curves and RFM
> are therefore **not computable** from this panel, and any "customer cohort" has
> to be a cohort of *stores* or *SKUs* instead.

> Every numeric claim in the prose below is printed by the cell it follows, so it
> can be checked against the source data.
>
> Where a claim is inferential, the test that supports it sits in the same section:
> HAC and rank trend tests in §2, a permutation test in §3, Fisher-z intervals with
> Benjamini-Hochberg in §4, a cluster bootstrap in §5.2, paired tests in §5.3.

## 1. Setup & data

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy import stats

sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.figsize"] = (13, 4.5)
plt.rcParams["axes.titlesize"] = 12
plt.rcParams["axes.titleweight"] = "bold"

DATA = Path("../data/dunnhumby - Breakfast at the Frat.xlsx")
pd.set_option("display.float_format", lambda v: f"{v:,.2f}")
pd.set_option("display.max_columns", 50)

print(f"pandas {pd.__version__} | seaborn {sns.__version__} | scipy {stats.__name__}")

In [ ]:
xl = pd.ExcelFile(DATA, engine="openpyxl")

tx = pd.read_excel(xl, "dh Transaction Data", header=1)
stores = pd.read_excel(xl, "dh Store Lookup", header=1)
products = pd.read_excel(xl, "dh Products Lookup", header=1)

# the store lookup ships 79 rows for 77 stores (4503 and 17627 twice, with a
# conflicting SEG_VALUE_NAME) -- dedupe once here, before any join
stores = stores.drop_duplicates("STORE_ID")
stores["SIZE_TIER"] = pd.qcut(stores.SALES_AREA_SIZE_NUM, 3,
                              labels=["small", "mid", "large"])

tx = tx.merge(products[["UPC", "DESCRIPTION", "CATEGORY", "MANUFACTURER"]],
              on="UPC", how="left")
tx = tx.merge(stores[["STORE_ID", "SEG_VALUE_NAME", "MSA_CODE", "SIZE_TIER"]],
              left_on="STORE_NUM", right_on="STORE_ID", how="left")

tx["ANY_PROMO"] = ((tx.DISPLAY == 1) | (tx.FEATURE == 1) | (tx.TPR_ONLY == 1)).astype(int)

# every WEEK_END_DATE is a Wednesday, so W-WED is an exact 7-day grid
tx["ISO_YEAR"] = tx.WEEK_END_DATE.dt.isocalendar().year.astype(int)
tx["ISO_WEEK"] = tx.WEEK_END_DATE.dt.isocalendar().week.astype(int)

print(f"tx: {tx.shape[0]:,} rows x {tx.columns.size} cols  "
      f"({tx.WEEK_END_DATE.nunique()} weeks, {tx.STORE_NUM.nunique()} stores, "
      f"{tx.UPC.nunique()} SKUs)")
print(f"panel: {tx.WEEK_END_DATE.min():%Y-%m-%d} -> {tx.WEEK_END_DATE.max():%Y-%m-%d}, "
      f"all weeks end on a {tx.WEEK_END_DATE.dt.day_name().iloc[0]}")

print("\nno customer-level key exists in any of the three sheets:")
print("  tx       :", list(tx.columns[:12]))
print("  stores   :", list(stores.columns[:9]))
print("  products :", list(products.columns))
print(f"  HHS <= VISITS in {(tx.HHS <= tx.VISITS).mean():.2%} of rows, "
      f"equal in {(tx.HHS == tx.VISITS).mean():.2%}, corr {tx.HHS.corr(tx.VISITS):.4f}")
print("  -> HHS/VISITS are per-row counts, so they cannot key a household across weeks")

print("\nweeks per calendar year:")
print(tx.groupby(tx.WEEK_END_DATE.dt.year).WEEK_END_DATE.nunique().to_frame("weeks").T)

print("\nstore strata:")
print(f"  SEG_VALUE_NAME: {stores.SEG_VALUE_NAME.value_counts().to_dict()}")
print(f"  MSA_CODE: {stores.MSA_CODE.nunique()} regions across "
      f"{stores.ADDRESS_STATE_PROV_CODE.nunique()} states")
print(stores.groupby("SIZE_TIER", observed=True).SALES_AREA_SIZE_NUM
      .agg(["size", "min", "max"]).round(0).T)

## 2. Trend — is the weekly series moving?

The distribution notebook read the weekly median and IQR as "flat 2009-2012". That
was true of the *median of a row*, which hides the aggregate. This section works on
the weekly **total** instead, which is the quantity a buyer would actually look at.

In [ ]:
weekly = (tx.groupby("WEEK_END_DATE")
            .agg(UNITS=("UNITS", "sum"),
                 SPEND=("SPEND", "sum"),
                 ROWS=("UPC", "size"),
                 PROMO_RATE=("ANY_PROMO", "mean"))
            .asfreq("W-WED"))

print(f"weekly grid: {len(weekly)} rows, {weekly.isna().sum().sum()} missing -> "
      f"{weekly.index.min():%Y-%m-%d} .. {weekly.index.max():%Y-%m-%d}")

t = np.arange(len(weekly))
print("\nOLS trend on the weekly total, and the naive end-to-end comparison:")
for col in ["UNITS", "SPEND"]:
    slope, intercept = np.polyfit(t, weekly[col], 1)
    per_year = slope * 52 / weekly[col].mean()
    e2e = weekly[col].iloc[-13:].mean() / weekly[col].iloc[:13].mean() - 1
    print(f"  {col:6s} mean {weekly[col].mean():>12,.0f} | slope {slope:>9,.1f}/wk "
          f"= {per_year:+7.2%}/yr | last 13 wks vs first 13 wks {e2e:+7.2%}")

iso_year = pd.Series(weekly.index.isocalendar().year.values, index=weekly.index)
print("\nweekly means by ISO year (2012 is a single week, so it is noise):")
print(weekly[["UNITS", "SPEND"]].groupby(iso_year).mean().round(0).T.to_string())
with pd.option_context("display.float_format", lambda v: f"{v:,.4f}"):
    print(weekly[["ROWS", "PROMO_RATE"]].groupby(iso_year).mean().T.to_string())

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(13, 7), sharex=True)
for ax, col in zip(axes, ["UNITS", "SPEND"]):
    y = weekly[col].values
    slope, intercept = np.polyfit(t, y, 1)
    ax.plot(t, y, color="steelblue", lw=0.9, label="weekly total")
    ax.plot(t, intercept + slope * t, color="crimson", lw=1.5, ls="--", label="OLS trend")
    ax.set_title(f"{col} per week")
    ax.set_ylabel(col)
    ax.legend(loc="upper left", fontsize=8)

axes[1].set_xticks(t[::13])
axes[1].set_xticklabels([d.strftime("%Y-%m") for d in weekly.index[::13]],
                        rotation=45, ha="right")
axes[1].set_xlabel("week ending")
fig.suptitle("Weekly totals and fitted trend", y=1.0, fontsize=13, fontweight="bold")
fig.tight_layout()

In [ ]:
# A slope through a series with an 81% seasonal swing is not a trend until it
# separates from noise. Three tests, because weekly data is autocorrelated and
# 156 points hold only three seasonal cycles:
#   HAC(4)          -- Newey-West standard errors on the raw series
#   season-adjusted -- the same slope with 51 week-of-year dummies, so the 52-week
#                      cycle is absorbed instead of being mistaken for drift
#   Mann-Kendall    -- rank correlation against time, no distributional assumption
def ols_slope(y, X, lag=4):
    """Slope of y on the first column of X, with Newey-West HAC(lag) errors."""
    X = np.column_stack([np.ones(len(y)), X])
    beta = np.linalg.lstsq(X, y, rcond=None)[0]
    resid = y - X @ beta
    XtX_inv = np.linalg.inv(X.T @ X)
    meat = (X * resid[:, None]).T @ (X * resid[:, None])
    for k in range(1, lag + 1):
        w = 1 - k / (lag + 1)
        G = (X[k:] * resid[k:, None]).T @ (X[:-k] * resid[:-k, None])
        meat += w * (G + G.T)
    return beta[1], np.sqrt((XtX_inv @ meat @ XtX_inv)[1, 1])


week_of_year = weekly.index.isocalendar().week.to_numpy()
D = pd.get_dummies(week_of_year, drop_first=True).to_numpy(dtype=float)

print("trend tests on the weekly total (p is for H0: slope = 0)")
for col in ["UNITS", "SPEND"]:
    y = weekly[col].to_numpy(dtype=float)
    b_raw, se_raw = ols_slope(y, t.reshape(-1, 1))
    b_adj, se_adj = ols_slope(y, np.column_stack([t, D]))
    mk = stats.kendalltau(t, y)
    print(f"  {col:6s} {b_raw:>9,.1f}/wk = {b_raw * 52 / y.mean():+7.2%}/yr | "
          f"p HAC(4) {2 * stats.t.sf(abs(b_raw / se_raw), len(y) - 2):.3f} | "
          f"p season-adj {2 * stats.t.sf(abs(b_adj / se_adj), len(y) - 2 - D.shape[1]):.3f} | "
          f"p Mann-Kendall {mk.pvalue:.3f}")

print("\nwith three tests on 156 autocorrelated weeks, the trend is only reported")
print("when all three reject; agreement in failure is a result too.")

**Value rises; volume does not fall in any way this panel can detect.** The `SPEND`
slope of +3.25%/yr survives every test applied to it — HAC(4) p = 0.048, week-of-year
dummies p = 0.037, Mann-Kendall p = 0.004 — so the dollar growth is real, if only just
under the parametric tests. `UNITS` is a different story: −0.21%/yr fails all three
(p = 0.92, 0.78 and 0.82), and three tests agreeing on failure is itself the result.
The 5.65% gap between the last 13 weeks and the first 13 is an endpoint comparison of
two different seasons, not evidence of decline; it stays in the table because it was
the original reading.

So the panel is neither flat nor shrinking. It is a flat-volume series with rising
value, which changes what later sections are allowed to claim: the promotion analysis
in §5.2 can explain price; it cannot explain a volume collapse that is not there.

The per-year means show how much of the movement is a level shift rather than smooth
drift: `UNITS` peaks in 2010 at 69,393 a week against 64,077 in 2009 and 64,472 in
2011, while `SPEND` never gives back its 2010 gain. 2012's 64,489 units and 190,698
dollars are a single week and should be ignored.

The promotion rate does **not** explain the spend growth — it is highest in 2009
(0.3051) and settles around 0.2722-0.2755 afterwards, so discounting is not being
increased to buy it. If anything the rate is flat to slightly down while value per unit
rises, which points at price and mix rather than promotion volume. §5.2 quantifies the
promotion lift properly.

## 3. Seasonality — decomposition

In [ ]:
def classical_decomp(y, period=52):
    '''Additive decomposition of a log1p series, no statsmodels required.

    trend    = centred moving average of length `period`
    seasonal = mean of the detrended values at each phase, mean-centred
    residual = what is left

    Needs at least two full periods to estimate the seasonal phase means; with 156
    weeks we have three, which is thin but workable.
    '''
    y = np.asarray(y, dtype=float)
    n = len(y)
    trend = pd.Series(y).rolling(period, center=True, min_periods=period // 2).mean()
    detrended = y - trend
    seasonal = np.array([np.nanmean(detrended[i::period]) for i in range(period)])
    seasonal -= seasonal.mean()
    seasonal = np.tile(seasonal, int(np.ceil(n / period)))[:n]
    return trend, seasonal, y - trend - seasonal


parts, rows = {}, []
for col in ["UNITS", "SPEND"]:
    y = np.log1p(weekly[col])
    trend, seasonal, resid = classical_decomp(y)
    parts[col] = (trend, seasonal, resid)
    total_var = np.var(y)
    amp = seasonal.max() - seasonal.min()
    rows.append({
        "trend_%": 100 * np.nanvar(trend) / total_var,
        "seasonal_%": 100 * np.var(seasonal) / total_var,
        "resid_%": 100 * np.nanvar(resid) / total_var,
        "seasonal_amp_log": amp,
        "peak_to_trough_%": 100 * np.expm1(amp),
        "resid_sd_log": np.nanstd(resid),
    })

decomp_tbl = pd.DataFrame(rows, index=["UNITS", "SPEND"])
with pd.option_context("display.float_format", lambda v: f"{v:,.2f}"):
    print(decomp_tbl.to_string())

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(13, 6), sharex=True)
for ax, col in zip(axes, ["UNITS", "SPEND"]):
    trend, seasonal, resid = parts[col]
    ax.plot(seasonal[:52], color="seagreen", lw=1.6)
    ax.axhline(0, color="gray", ls=":", lw=1)
    ax.set_title(f"{col} — seasonal profile, phase-averaged over all cycles")
    ax.set_ylabel("log effect")

axes[1].set_xlabel("week of the cycle (1 = first week of the panel)")
fig.suptitle("Seasonal component of the weekly log series", y=1.0,
             fontsize=13, fontweight="bold")
fig.tight_layout()

In [ ]:
# Is the seasonal profile more than phase-averaging three cycles produces on its
# own? Shuffle the detrended series against the week labels: the phase means then
# carry no calendar information, and the share they reach is the noise floor.
def seasonal_share(detrended, period=52):
    phase = np.array([np.nanmean(detrended[i::period]) for i in range(period)])
    phase -= phase.mean()
    return np.var(phase)


rng = np.random.default_rng(0)
rows = []
for col in ["UNITS", "SPEND"]:
    y = np.log1p(weekly[col].to_numpy(dtype=float))
    trend = pd.Series(y).rolling(52, center=True, min_periods=26).mean().to_numpy()
    detrended = y - trend
    observed = seasonal_share(detrended) / np.var(y)
    null = np.array([seasonal_share(rng.permutation(detrended)) / np.var(y)
                     for _ in range(2_000)])
    rows.append({"series": col, "seasonal_share": observed,
                 "shuffle_median": np.median(null),
                 "p (2,000 shuffles)": np.mean(null >= observed)})

with pd.option_context("display.float_format", lambda v: f"{v:,.4f}"):
    print(pd.DataFrame(rows).set_index("series").to_string())

# the coarser view that made the cycle look absent, for comparison
month_share = tx.groupby(tx.WEEK_END_DATE.dt.month).UNITS.sum() / tx.UNITS.sum()
name_of = lambda m: pd.Timestamp(2000, int(m), 1).strftime("%B")
print(f"\ncalendar-month share of all units: {month_share.min():.1%} "
      f"({name_of(month_share.idxmin())}) to {month_share.max():.1%} "
      f"({name_of(month_share.idxmax())})")

**There is a seasonal cycle, and it is larger than phase-averaging alone produces.**
The seasonal term carries 43.07% of `UNITS` variance against 6.83% for the trend, with a
peak-to-trough swing of 81.21%. For `SPEND` the seasonal share is 39.86% against 9.98%,
swinging 56.57%. The variance share by itself proves little: shuffling the detrended
series against its week labels 2,000 times already yields a median share of 0.3037 and
0.2882. What separates the real cycle from that floor is the permutation test, which
puts the observed share above the shuffle distribution at p = 0.0050 for `UNITS` and
p = 0.0120 for `SPEND`.

TODO: why is the seasonality so large?

This is worth flagging against the earlier notebooks. Calendar-month totals of `UNITS`
vary only between 7.1% (October) and 9.9% (December) of the panel total, which reads as
"no seasonality". That aggregation is too coarse: it averages the within-month weekly
peak and trough into each other. The 156-week series sees the actual cycle.

The residual term takes 49.01% and 48.03% of variance respectively, which sounds like a
lot but is expected here: a single centred 52-week moving average is a weak trend
estimator, so most of what it cannot explain is pushed into the residual rather than
recovered. Residual **standard deviation** is only 0.12 and 0.10 in log units — about
±10-12% week to week, which is ordinary noise for a grocery staple. I would not read
the variance split as a quality score; read the amplitude and the residual scale.

## 4. Repeat-year cohorts — the same ISO week, 2010 against 2011

In [ ]:
iso_coverage = tx.groupby("ISO_YEAR").WEEK_END_DATE.nunique()
print("ISO weeks covered per year:")
print(iso_coverage.to_frame("iso_weeks").T)

full_years = iso_coverage[iso_coverage == 52].index.tolist()
print(f"\nfull 52-week years: {full_years}  ->  "
      f"only {len(full_years) - 1} usable year-over-year pair")
print("2009 is short 1 week and 2012 holds a single week, so neither can be paired "
      "or averaged against a full year.")

pair = tx[tx.ISO_YEAR.isin(full_years)]
iso_tables = {}

for stratum in ["SEG_VALUE_NAME", "SIZE_TIER", "MSA_CODE"]:
    wide = (pair.groupby([stratum, "ISO_WEEK", "ISO_YEAR"], observed=True)
                .SPEND.mean().unstack("ISO_YEAR").dropna())
    iso_tables[stratum] = wide

    rows = [{"stratum": name, "n_paired_weeks": len(d),
             f"corr_{full_years[0]}_{full_years[1]}": d[full_years[0]].corr(d[full_years[1]]),
             f"mean_ratio_{full_years[1]}/{full_years[0]}":
                 (d[full_years[1]] / d[full_years[0]]).mean()}
            for name, d in wide.groupby(level=0)]
    weeks_paired = wide.groupby(level=0).size().iloc[0]
    print(f"\n=== {stratum} — {weeks_paired} paired ISO weeks, n = {len(wide)} rows ===")
    with pd.option_context("display.float_format", lambda v: f"{v:,.3f}"):
        print(pd.DataFrame(rows).set_index("stratum").to_string())

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5), sharey=True)
for ax, seg in zip(axes, sorted(pair.SEG_VALUE_NAME.dropna().unique())):
    d = (pair[pair.SEG_VALUE_NAME == seg]
           .groupby(["ISO_WEEK", "ISO_YEAR"]).SPEND.mean()
           .unstack("ISO_YEAR").dropna())
    for yr, colour, ls in [(full_years[0], "steelblue", "-"),
                           (full_years[1], "crimson", "--")]:
        ax.plot(d.index, d[yr] / d[yr].mean(), color=colour, lw=1.4, ls=ls, label=str(yr))
    ax.set_title(f"{seg}   corr {d[full_years[0]].corr(d[full_years[1]]):+.2f}")
    ax.set_xlabel("ISO week")

axes[0].set_ylabel("spend, each year indexed to its own mean")
axes[0].legend(fontsize=8)
fig.suptitle(f"Repeat-year cohorts — same ISO week, {full_years[0]} vs {full_years[1]}",
             y=1.02, fontsize=13, fontweight="bold")
fig.tight_layout()

In [ ]:
# n = 52 paired weeks, so a correlation needs an interval before it can be
# ranked against another one. Fisher z per stratum, then Benjamini-Hochberg
# across every stratum tested in this section.
def fisher_ci(r, n, alpha=0.05):
    z = np.arctanh(r)
    half = stats.norm.ppf(1 - alpha / 2) / np.sqrt(n - 3)
    return np.tanh(z - half), np.tanh(z + half)


rows = []
for stratum, wide in iso_tables.items():
    for name, d in wide.groupby(level=0):
        r, n = d[full_years[0]].corr(d[full_years[1]]), len(d)
        lo, hi = fisher_ci(r, n)
        t_stat = r * np.sqrt((n - 2) / max(1 - r ** 2, 1e-12))
        rows.append({"stratum": stratum, "group": str(name), "r": r,
                     "ci_low": lo, "ci_high": hi,
                     "p": 2 * stats.t.sf(abs(t_stat), n - 2)})

repeat_tbl = pd.DataFrame(rows).sort_values("p").reset_index(drop=True)
m = len(repeat_tbl)
repeat_tbl["q"] = (repeat_tbl.p * m / (repeat_tbl.index + 1)).cummax().clip(upper=1)

show = repeat_tbl[["stratum", "group", "r", "ci_low", "ci_high"]].copy()
show["p"] = repeat_tbl.p.map(lambda v: f"{v:.1e}")
show["q (BH)"] = repeat_tbl.q.map(lambda v: f"{v:.1e}")
show["excludes 0"] = np.where(repeat_tbl.ci_low > 0, "yes", "no")

print(f"{m} strata tested, n = 52 paired ISO weeks each")
with pd.option_context("display.float_format", lambda v: f"{v:,.3f}"):
    print(show.to_string(index=False))
print(f"\nintervals excluding zero: {(repeat_tbl.ci_low > 0).sum()} of {m}"
      f" | surviving Benjamini-Hochberg at q < 0.05: {(repeat_tbl.q < 0.05).sum()} of {m}")

**Seasonality repeats more than the first draft allowed, but only moderately, and the
gaps between strata are not established.** Across 52 paired ISO weeks, 9 of the 15
strata have a 2010-vs-2011 correlation whose 95% interval excludes zero, and 8 survive
Benjamini-Hochberg at q < 0.05. The strongest is `VALUE` at 0.521 [0.289, 0.695],
followed by the mid size tier at 0.408 [0.153, 0.613] and the MSAs in the 0.34-0.45
band. Nine intervals excluding zero is far more than the one false positive that 15
tests at the 5% level would produce, so the repetition is real.

What does not repeat is the mass market: `MAINSTREAM` 0.179 [−0.099, 0.431], `UPSCALE`
0.218 [−0.058, 0.464], the large stores 0.144 [−0.134, 0.401] and `MSA 47540` at
−0.000 [−0.273, 0.273]. `MAINSTREAM` is 43 of the 77 stores, which is why a pooled
"same week last year" baseline looks weak even where it works for `VALUE`.

The ranking between strata is the part not to over-read: `VALUE`'s lower bound (0.289)
sits below the mid tier's point estimate (0.408), and every interval above 0.4 overlaps
`VALUE`'s. Ordering the strata by point estimate is fine; claiming `VALUE` repeats
better than the mid tier is not supported by 52 observations.

The segment level ratios are the reassuring part: mean 2011 against 2010 lands between
1.003 and 1.044 for all three segments, so no segment cohort is meaningfully growing or
shrinking year on year. It is the *shape within the year* that is only partly stable,
not the level. The size tiers are looser (0.994 to 1.035) and the MSAs looser still,
with `MSA 17780` at 0.903.

So the honest reading: seasonality is real (§3) and modestly repeatable — enough for a
"same week last year" baseline at `VALUE` and several MSAs, not enough to rely on for
`MAINSTREAM`, `UPSCALE` or the large stores.

TODO open question for me. Why is the seasonality so unpredictable

## 5. Derived metrics — what can actually be built

Sections 2-4 were about the time axis and the strata. This section builds the four
metrics the brief asks for and states, for each, where it stops being trustworthy.

### 5.1 Price elasticity of demand

In [ ]:
from sklearn.linear_model import LinearRegression

el = tx[(tx.PRICE > 0) & (tx.BASE_PRICE > 0)].copy()
el["log_p"] = np.log(el.PRICE)
el["log_u"] = np.log(el.UNITS + 1)


def fit(y, X, names):
    '''OLS with a standard-error column: the sklearn fit supplies the coefficients
    and the intercept, and (X'X)^-1 is rebuilt against an explicit constant to
    get the standard errors.'''
    m = LinearRegression().fit(X, y)
    resid = y - m.predict(X)
    D = np.column_stack([*X.to_numpy().T, np.ones(len(X))])
    cov = np.linalg.inv(D.T @ D) * (resid @ resid) / (len(y) - D.shape[1])
    se = np.sqrt(np.diag(cov))
    beta = np.r_[m.coef_, m.intercept_]
    return pd.DataFrame({"coef": beta, "std_err": se, "t": beta / se}, index=names)


def within(df, keys):
    '''De-mean log price, log units and promo within the given grouping keys, so the
    coefficient is identified off within-group variation only.'''
    out = df.copy()
    g = out.groupby(keys, observed=True)
    for c in ["log_p", "log_u", "ANY_PROMO"]:
        out[c] = out[c] - g[c].transform("mean")
    return out


specs = {"pooled (cross-sectional)": None, "within UPC": ["UPC"],
         "within UPC x STORE": ["UPC", "STORE_NUM"]}

for label, keys in specs.items():
    d = el if keys is None else within(el, keys)
    tab = fit(d.log_u, d[["log_p", "ANY_PROMO"]], ["log(PRICE)", "ANY_PROMO", "const"])
    print(f"\n=== {label} ===")
    with pd.option_context("display.float_format", lambda v: f"{v:,.4f}"):
        print(tab.to_string())
    print(f"  elasticity {tab.loc['log(PRICE)', 'coef']:+.4f}"
          f" | promo uplift {np.expm1(tab.loc['ANY_PROMO', 'coef']):+.2%}")

**The sign is robust and the magnitude is not, and the reason is simultaneity, not
sloppiness.** The pooled cross-sectional fit gives −0.7876, controlling for a promotion
uplift of +39.86%. Adding UPC fixed effects takes it to −0.9950. Adding store as well
takes it to −1.6608 — a unit-elastic estimate — and shrinks the promo uplift to +22.59%.

Getting *more* negative as controls are added is the tell. Within a single SKU in a
single store, the weeks when the shelf price is high are the weeks when the item is
not promoted, and non-promotional weeks cluster in periods of normal or high demand.
Price here is set in response to expected demand, so it is an endogenous regressor and
no specification in this table isolates a causal elasticity. The honest summary is
"clearly price-elastic and negative, somewhere between −0.8 and −1.7 depending on how
much endogeneity you refuse to absorb".

The promotion coefficient is the more trustworthy number here and it is large: a
promotion is associated with roughly a third more units on the pooled fit.

### 5.2 Promotion lift vs discount — a proxy, not ROI

In [ ]:
pr = el.copy()
pr["DISCOUNT"] = (pr.BASE_PRICE - pr.PRICE).clip(lower=0)
pr["DEPTH"] = pr.DISCOUNT / pr.BASE_PRICE
pr["DISC_SPEND"] = pr.DISCOUNT * pr.UNITS

promo_tbl = (pr.groupby("ANY_PROMO")
               .agg(rows=("UPC", "size"), spend=("SPEND", "sum"), units=("UNITS", "sum"),
                    med_depth=("DEPTH", "median"), med_price=("PRICE", "median"),
                    med_base=("BASE_PRICE", "median"), med_units=("UNITS", "median"),
                    disc_given_up=("DISC_SPEND", "sum"))
               .rename(index={0: "No promo", 1: "On promo"}))
print(promo_tbl.round(2).T)

print(f"\nmedian discount depth on promo rows: "
      f"{pr.loc[pr.ANY_PROMO == 1, 'DEPTH'].median():.2%} "
      f"(mean {pr.loc[pr.ANY_PROMO == 1, 'DEPTH'].mean():.2%})")
print(f"share of rows on promotion: {pr.ANY_PROMO.mean():.2%}")

# paired within SKU-store cells: promo weeks against that cell's own non-promo weeks
cell = (pr.groupby(["UPC", "STORE_NUM", "ANY_PROMO"])
           .agg(units=("UNITS", "mean"), spend=("SPEND", "mean"), n=("UNITS", "size"))
           .unstack("ANY_PROMO"))
usable = (cell[("units", 0)] > 0) & (cell[("n", 0)] >= 4) & (cell[("n", 1)] >= 1)
unit_lift = cell[("units", 1)][usable] / cell[("units", 0)][usable]
spend_lift = cell[("spend", 1)][usable] / cell[("spend", 0)][usable]

print(f"\npaired SKU-store cells with >=4 non-promo weeks and >=1 promo week: "
      f"{usable.sum():,} of {len(usable):,}")
print(f"median unit lift on promo  {unit_lift.median():+.1%}")
print(f"median spend lift on promo {spend_lift.median():+.1%}")

# each promo row gets credited with what it would have sold at its own cell's
# non-promo mean, marked up by the median observed lift
baseline = cell[("spend", 0)][usable]
cell_lookup = pd.Series(baseline[baseline > 0].values, index=baseline[baseline > 0].index)
promo_rows = pr[pr.ANY_PROMO == 1].set_index(["UPC", "STORE_NUM"])
credit = pd.Series(promo_rows.index.map(cell_lookup), index=promo_rows.index)
matched = credit.notna()
incr_spend = (credit[matched] * (spend_lift.median() - 1)).sum()
discount_cost = pr.loc[pr.ANY_PROMO == 1, "DISC_SPEND"].sum()

print(f"\npromo rows matched to a usable baseline cell: {matched.sum():,} of {len(promo_rows):,}")
print(f"spend given up in discounting     ${discount_cost:>14,.0f}")
print(f"incremental spend at median lift ${incr_spend:>14,.0f}")
print(f"proxy ratio (revenue / discount) {incr_spend / discount_cost:>14,.2f}x")
print("\nNOTE: no cost, margin or COGS field exists in this dataset, so a true ROI")
print("cannot be computed. The ratio above is revenue-lift against discount-given-up.")
print("Promotion weeks are also not randomly assigned, so the lift is not causal.")

In [ ]:
# The ratio has no interval. Bootstrap over cells, not rows: weeks inside a
# SKU-store cell are correlated, so resampling rows would understate the spread
# and make 0.86x look more certain than it is.
disc_per_cell = (pr.loc[pr.ANY_PROMO == 1]
                   .groupby(["UPC", "STORE_NUM"]).DISC_SPEND.sum()
                   .reindex(baseline.index).fillna(0.0))
boot = pd.DataFrame({"baseline": baseline,
                     "n_promo": cell[("n", 1)][usable],
                     "lift": spend_lift,
                     "discount": disc_per_cell}).dropna()

b = boot.baseline.to_numpy()
n_promo = boot.n_promo.to_numpy()
lift_v = boot.lift.to_numpy()
disc_v = boot.discount.to_numpy()

rng = np.random.default_rng(0)
B = 2_000
draws = np.empty(B)
for i in range(B):
    idx = rng.integers(0, len(boot), len(boot))
    draws[i] = ((np.median(lift_v[idx]) - 1) * (b[idx] * n_promo[idx]).sum()
                / disc_v[idx].sum())

ratio = (np.median(lift_v) - 1) * (b * n_promo).sum() / disc_v.sum()
lo, hi = np.percentile(draws, [2.5, 97.5])

print(f"cells in the bootstrap: {len(boot):,} | draws: {B:,}")
print(f"ratio {ratio:.2f}x  95% CI [{lo:.3f}, {hi:.3f}]  "
      f"P(ratio < 1) = {np.mean(draws < 1):.3f}")
print("the interval covers cell-to-cell variation only -- not the choice of")
print("crediting rule, and not profit.")

**Promotion is the mechanism behind price, and on this estimator it does not pay for
itself.** 28.43% of rows carry a promotion flag at a median discount depth of 19.05%;
median price falls from $3.17 off-promo to $2.50 on-promo while median base price is
essentially unchanged ($3.17 against $3.19). Median units go 9 to 16 — a **ratio of
1.83x**, not a lift of +183.4%. The spend ratio is 1.48x.

The headline number is the ratio: $3,576,964 of discount given up against $3,073,881 of
incremental spend, **0.86x**. A cluster bootstrap over the 3,857 usable SKU-store cells
puts the 95% interval at [0.814, 0.906] with P(ratio < 1) = 1.000, so the ratio is well
separated from break-even *within this estimator*. That qualifier carries the weight:
the interval measures cell-to-cell variation, not the choice of crediting rule, and it
says nothing about profit — there is no cost field to turn revenue into margin.

Three warnings belong next to that number. First, with no cost field in the dataset, a
60%-margin item and a 20%-margin item score identically, so this is revenue against
discount and never a return on investment. Second, the paired lift is confounded:
promotions are scheduled into weeks the retailer already expects to be strong, and it is
those weeks that show the larger lift. The elasticity result in §5.1 is the same problem
seen from the other side. Third, applying one median lift to every cell ignores
SKU-to-SKU variation in response, so 0.86x is an order-of-magnitude figure with a
cell-level interval, not a point estimate of anything.

### 5.3 Basket economics and the active-household proxy

In [ ]:
# the basket metrics need a store-week table, and §5.3 groups it by segment, so
# the store lookup has to come in here
sw = (tx.groupby(["STORE_NUM", "WEEK_END_DATE"])
        .agg(spend=("SPEND", "sum"), units=("UNITS", "sum"),
             baskets=("VISITS", "sum"), hhs=("HHS", "sum"))
        .reset_index())
sw = sw.merge(stores[["STORE_ID", "SEG_VALUE_NAME"]],
              left_on="STORE_NUM", right_on="STORE_ID", how="left")
sw["per_basket"] = sw.spend / sw.baskets
sw["per_household"] = sw.spend / sw.hhs
sw["units_per_basket"] = sw.units / sw.baskets

print(f"store-week rows: {len(sw):,} across {sw.STORE_NUM.nunique()} stores and "
      f"{sw.WEEK_END_DATE.nunique()} weeks")

In [ ]:
print("store-week aggregates (the unit of observation for basket metrics):")
print(sw[["spend", "per_basket", "per_household", "units_per_basket"]]
      .describe().loc[["mean", "50%", "min", "max"]].round(3).T)

print("\nby segment:")
print(sw.groupby("SEG_VALUE_NAME")[["spend", "per_basket", "per_household",
                                   "units_per_basket"]].median().round(3).T)

print("\nby ISO year:")
sw["ISO_YEAR"] = sw.WEEK_END_DATE.dt.isocalendar().year.astype(int)
year_tbl = sw.groupby("ISO_YEAR")[["spend", "per_basket", "per_household",
                                   "units_per_basket"]].median().round(3)
# 3dp matters here: 2dp turns 1.135 -> 1.14 and doubles the apparent decline
with pd.option_context("display.float_format", lambda v: f"{v:,.3f}"):
    print(year_tbl.T)

print("\nthe WAH proxy, and why it is only a proxy:")
store_week_hhs = tx.groupby(["STORE_NUM", "WEEK_END_DATE"]).HHS.sum()
print(f"  sum of HHS across UPCs, median store-week: {store_week_hhs.median():,.0f}")
print(f"  share of SKU-rows where a household equals a basket: "
      f"{(tx.HHS == tx.VISITS).mean():.2%}")
print("  HHS is measured per SKU, so summing it counts a household once per SKU it")
print("  bought. Use it as an engagement proxy, never as a household count.")

In [ ]:
# The medians above are descriptive. Pair each store against itself, 2009 against
# 2011, so the comparison is within store and both sides share the calendar.
sw["ISO_YEAR"] = sw.WEEK_END_DATE.dt.isocalendar().year.astype(int)
paired = (sw[sw.ISO_YEAR.isin([2009, 2011])]
            .groupby(["STORE_NUM", "ISO_YEAR"])[["per_basket", "units_per_basket"]]
            .median().unstack("ISO_YEAR"))

print("within-store change, 2009 -> 2011")
for col in ["per_basket", "units_per_basket"]:
    before, after = paired[(col, 2009)], paired[(col, 2011)]
    change = (after - before).dropna()
    tt = stats.ttest_rel(after, before, nan_policy="omit")
    wil = stats.wilcoxon(after, before, nan_policy="omit")
    print(f"  {col:>17s} {change.median():+.4f} "
          f"({change.median() / before.median():+.2%}) | n = {len(change)} stores "
          f"| t p = {tt.pvalue:.1e} | Wilcoxon p = {wil.pvalue:.1e}")

**The ticket grows on price, not on volume.** Median spend per basket climbs $3.055 →
$3.090 → $3.244 across 2009-2011 and median units per basket slips 1.135 → 1.133 →
1.125. Both moves hold within store: pairing each store's 2009 and 2011 medians, spend
per basket rises $0.20 (+6.53%) at t p = 6.4e-38, Wilcoxon p = 2.6e-14, while units per
basket falls 0.0082 (−0.72%) at t p = 8.5e-09, Wilcoxon p = 1.9e-07. The volume change
is statistically unmistakable and economically tiny; the ticket is carried by price.

Per-household spend tracks per-basket spend slightly above it ($3.118 against $3.055 in
2009, $3.308 against $3.244 in 2011), which is the arithmetic consequence of the 81.88% of
rows where one household buys exactly one basket's worth of that SKU. The two measures
would diverge sharply if basket size fell further, so they are worth keeping separate.

The `WAH` figure of 630 is the closest thing here to an active-user count, and it is
weak: `HHS` is recorded per SKU, so a household buying four SKUs in a week is counted
four times. It is usable as an engagement trend, not as a customer count, and this
notebook will not present it as one.

### 5.4 Distribution penetration

In [ ]:
# category-level penetration is uninformative here -- check that before using it
cat_stores = tx.groupby("CATEGORY").STORE_NUM.nunique()
print(f"stores in the panel: {tx.STORE_NUM.nunique()}")
print("category penetration (share of panel stores carrying the category):")
print((cat_stores / tx.STORE_NUM.nunique()).round(3).to_frame("penetration").T)
print("-> saturated, so measure penetration per UPC instead")

pen = (tx.groupby(["WEEK_END_DATE", "UPC"]).STORE_NUM.nunique().rename("stores")
         .reset_index())
pen["penetration"] = pen.stores / tx.STORE_NUM.nunique()
pen = pen.merge(products[["UPC", "DESCRIPTION", "CATEGORY"]], on="UPC")

penu = pen.groupby(["UPC", "DESCRIPTION", "CATEGORY"]).penetration.agg(
    ["mean", "min", "max"]).sort_values("mean")
print("\nUPC distribution penetration, lowest 6 and highest 3:")
print(penu.head(6).round(3).T)
print(penu.tail(3).round(3).T)

tenure = (tx.groupby("UPC").WEEK_END_DATE
            .agg(first="min", last="max", weeks="nunique")
            .join(products.set_index("UPC")[["DESCRIPTION", "MANUFACTURER"]]))
print(f"\nUPCs present in fewer than 156 weeks: {(tenure.weeks < 156).sum()} of {len(tenure)}")
print(tenure[tenure.weeks < 156].to_string())

cat_month = (tx.groupby([tx.WEEK_END_DATE.dt.to_period("M"), "CATEGORY"]).SPEND.sum().unstack())
share = cat_month.div(cat_month.sum(axis=1), axis=0)
print("\ncategory share of spend, first month vs last month:")
with pd.option_context("display.float_format", lambda v: f"{v:,.2f}"):
    print((100 * pd.DataFrame({"first": share.iloc[0], "last": share.iloc[-1]}).T).to_string())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 4.5))
shown = penu.head(6).index.get_level_values(0)
for upc, colour in zip(shown, ["steelblue", "seagreen", "crimson", "darkorange",
                               "purple", "brown"]):
    d = pen[pen.UPC == upc].set_index("WEEK_END_DATE").penetration
    axes[0].plot(d.index, d, color=colour, lw=1.3,
                 label=products.set_index("UPC").DESCRIPTION.get(upc, str(upc))[:26])

axes[0].set_title("distribution penetration of the 6 least-distributed UPCs")
axes[0].set_ylabel("share of the 77 stores carrying the SKU")
axes[0].legend(fontsize=7, title="UPC", title_fontsize=7)
axes[0].set_xticks(pen.WEEK_END_DATE.unique()[::26])
axes[0].set_xticklabels([pd.Timestamp(d).strftime("%Y-%m")
                         for d in pen.WEEK_END_DATE.unique()[::26]], rotation=45, ha="right")

axes[1].plot(share.index.astype(str), share["COLD CEREAL"], color="steelblue", lw=1.5, label="COLD CEREAL")
axes[1].plot(share.index.astype(str), share["FROZEN PIZZA"], color="seagreen", lw=1.5, label="FROZEN PIZZA")
axes[1].plot(share.index.astype(str), share["BAG SNACKS"], color="darkorange", lw=1.5, label="BAG SNACKS")
axes[1].plot(share.index.astype(str), share["ORAL HYGIENE PRODUCTS"], color="crimson", lw=1.5, label="ORAL HYGIENE")
axes[1].set_title("category share of monthly spend")
axes[1].set_ylabel("share of spend")
axes[1].legend(fontsize=7)
axes[1].set_xticks(range(0, len(share), 6))
axes[1].set_xticklabels([str(p) for p in share.index[::6]], rotation=45, ha="right")

fig.suptitle("Penetration and category mix", y=1.02, fontsize=13, fontweight="bold")
fig.tight_layout()

**Category penetration is saturated, but UPC distribution is not.** All four categories
sit in all 77 stores from the first week, so category-level penetration carries no
information. At UPC level the spread is wide: the least-distributed SKUs never clear
the whole panel, and their weekly penetration lines move independently of each other.
That is the useful version of the question — a SKU's reach is a store-level assortment
decision, not a demand outcome.

Eight of 55 UPCs do not span the full 156 weeks, and those eight are the real
structural events in the panel: the three `KING` frozen pizzas enter on 2010-02-24 and
run to the end, while `COLG SPEARMINT WISP` appears on 2009-03-18 and **disappears on
2011-03-30**. Four `POST` cereals and one `QUAKER` item enter during 2009. Delisting
and launch are both visible here.

Category mix shifts over the panel: `COLD CEREAL` gives up share (46.11% to 43.86%)
while `FROZEN PIZZA` takes it (26.30% to 34.05%), and `BAG SNACKS` and `ORAL HYGIENE
PRODUCTS` both decline. The frozen-pizza gain tracks the `KING` entry, though the first
and last month are two points and should not be read as a trend on their own.


## 7. Conclusion

**Is the weekly series moving?** In value, yes; in volume, no. `SPEND` trends +3.25%/yr
and passes all three trend tests (HAC p = 0.048, season-adjusted p = 0.037,
Mann-Kendall p = 0.004). `UNITS` gives −0.21%/yr and fails all three (p = 0.92, 0.78,
0.82), so the "shrinking volume" reading does not survive: the 5.65% endpoint gap is two
different seasons. §2 corrects both `1_distribution_analysis.ipynb` (which read a flat
weekly median) and its own first draft (which read a volume decline): the weekly median
is flat, and the weekly total is flat in volume too — only the dollar total moves.

**Is there a seasonal cycle, and does it repeat?** The cycle is real and large —
81.21% peak-to-trough on `UNITS`, 56.57% on `SPEND` — and a permutation test puts the
seasonal variance share above the shuffle floor (p = 0.0050 and 0.0120). It repeats more
than the first draft claimed but not strongly: 9 of 15 strata have a 2010-vs-2011
interval that excludes zero, 8 survive Benjamini-Hochberg, and the coefficients are
modest (0.34-0.52). `MAINSTREAM`, `UPSCALE`, the large stores and `MSA 47540` do not
repeat at all, and the intervals overlap too widely to rank strata against each other.

**Do strata behave differently over time?** On level, yes and stably: the three segments
hold their relative positions across the panel and their 2011-to-2010 mean spend ratios all
sit between 1.003 and 1.044. On shape, `VALUE` and the mid size tier repeat at 0.521
and 0.408 while `MAINSTREAM` and `UPSCALE` do not — but those intervals overlap, so the
difference between strata is not established. Basket behaviour splits them cleanly —
`VALUE` has the lowest store-week spend but the highest spend per basket at 3.21,
`UPSCALE` the highest volume of business. `MAINSTREAM` is 43 of the 77 stores, so the
pooled numbers in §2 are mostly a `MAINSTREAM` story.

**Which metrics can be built here?** Elasticity is computable but not causally
identified — the pooled −0.7876, within-UPC −0.9950 and within-UPC-x-store −1.6608
differ because price is set in response to demand, so I would not quote any single one.
Promotion economics are the most interesting output and the most sobering: $3,576,964 of
discount against $3,073,881 of incremental spend, a **0.86x** ratio with a cell-level
interval of [0.814, 0.906] — separated from break-even on this estimator, and still not
a profit statement, because the dataset has no cost field. Basket economics are the
cleanest result here — spend per basket $3.055 to $3.244 while units per basket falls
1.135 to 1.125, both significant within store, which identifies price as the growth
mechanism rather than basket size. UPC penetration and the eight entry/exit UPCs are the
most actionable structural findings.

**Open threads.**

- Convert the 0.86x revenue ratio into a profit statement. It cannot be done without a
  cost assumption, but a break-even margin can be derived and reported as a threshold
  rather than a point.
- Test the payback per category and per store segment. If `VALUE` and `UPSCALE` have
  different paybacks the pooled number is hiding a real decision.
- Identify price causally, or stop quoting an elasticity. An instrument or a
  difference-in-differences around the `KING` launch (§5.4) would break the simultaneity
  that §5.1 runs into.
- The `COLG SPEARMINT WISP` delisting on 2011-03-30 is a natural experiment on
  category-level cannibalisation between the four oral-hygiene UPCs — it is the only
  clean exit event in the panel.
- Re-run §3 on the segment panels. One pooled seasonal cycle across 77 stores may be
  hiding three different ones, and the `VALUE` segment's 0.521 repeatability is a hint
  that it is.
- The `HHS` double-count in §5.3 can be bounded rather than assumed — a household
  cannot exceed `AVG_WEEKLY_BASKETS` per store, which gives an upper bound on the
  over-count.
- The decomposition uses a single centred moving average because `statsmodels` is not a
  dependency. A robust STL would be worth adding if the seasonal profile is going to be
  used for forecasting rather than description.